In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from wq_model import Config, WaterQualityDataset, load_and_prepare_dataframe, load_submission_for_inference, MultiModalResNetRegressor, train_with_early_stopping, predict

In [2]:
# Setup configuration
Config.set_seed()
device = Config.get_device()
print("Device:", device)

Device: cuda


In [3]:

Config.INDEX_BANDS = [6, 8, 7, 9]  # [NDVI, MNDWI, NDMI, HydroRIVERS weighted]

Config.USE_IMAGES = True
Config.USE_TABULAR = True
Config.USE_DATETIME_FEATURES = True

    # Target columns
Config.TARGET_COLS = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus",
]

# Data filtering
Config.MAX_DAYS_DIFF = 120
Config.CLOUD_THRESHOLD_PCT = 20

# Cross-validation settings
Config.N_FOLDS = 1
Config.TEST_SIZE = 0.2

# Training hyperparameters
Config.BATCH_SIZE = 8
Config.NUM_WORKERS = 6
Config.MAX_EPOCHS = 40
Config.PATIENCE = 8
Config.LR = 2e-4
Config.WEIGHT_DECAY = 2e-3

# Model hyperparameters
Config.TAB_DROPOUT = 0.3
Config.HEAD_DROPOUT = 0.4


Config.print_config()

__module__: wq_model.config
__doc__: Configuration class for water quality regression
SEED: 42
ROOT: .
CSV_PATH: data/water_quality_training_dataset.csv
SOILGRIDS_CSV_PATH: data/soilgrids_features_full.csv
TC_FEATURES_ALL_CSV_PATH: data/tc_features_all.csv
META_PATH: data/river_patches/metadata.json
IDX_DIR: data/river_patches_with_indices
VALID_PATCHES_CACHE: data/river_patches/valid_patches.json
LANDCOVERAGE_DIR: data_injestion/landcoerage location jsons
CLOUD_COVERAGE_CACHE: None
VALID_INDEXES_CACHE: None
INDEX_BANDS: [6, 8, 7, 9]
IMG_SIZE: 224
USE_IMAGES: True
USE_TABULAR: True
USE_DATETIME_FEATURES: True
INCLUDE_LANDCOVERAGE_CHANNEL: False
TARGET_COLS: ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
MAX_DAYS_DIFF: 120
CLOUD_THRESHOLD_PCT: 20
N_FOLDS: 1
TEST_SIZE: 0.2
BATCH_SIZE: 8
NUM_WORKERS: 6
MAX_EPOCHS: 40
PATIENCE: 8
LR: 0.0002
WEIGHT_DECAY: 0.002
TAB_DROPOUT: 0.3
HEAD_DROPOUT: 0.4
set_seed: <classmethod(<function Config.set_seed at 0x7f6b76f0b

In [4]:
# Load and prepare data
df, TAB_COLS = load_and_prepare_dataframe(
    csv_path=Config.CSV_PATH,
    meta_path=Config.META_PATH,
    idx_dir=Config.IDX_DIR,
    target_cols=Config.TARGET_COLS,
    index_bands=Config.INDEX_BANDS,
    use_datetime_features=Config.USE_DATETIME_FEATURES,
    max_days_diff=Config.MAX_DAYS_DIFF,
    valid_patches_cache=Config.VALID_PATCHES_CACHE,
    cloud_threshold_pct=Config.CLOUD_THRESHOLD_PCT,
    cloud_coverage_cache=Config.CLOUD_COVERAGE_CACHE,
    valid_indexes_cache=Config.VALID_INDEXES_CACHE,
)

# Choose transform per target: "none" or "log1p"
TARGET_TRANSFORMS = {
    "Total Alkalinity": "none",
    "Electrical Conductance": "none",
    "Dissolved Reactive Phosphorus": "log1p",
}

for t in Config.TARGET_COLS:
    transform_name = TARGET_TRANSFORMS.get(t, "none")
    if transform_name == "log1p":
        df[t] = np.log1p(df[t])
    elif transform_name != "none":
        raise ValueError(f"Unsupported transform '{transform_name}' for target '{t}'")

print("--------------------------------")
print("Total usable samples:", len(df))
print("Unique stations/groups:", df["group_id"].nunique())
print("Targets:", Config.TARGET_COLS)
print("Using datetime features:", Config.USE_DATETIME_FEATURES)
print("Max days difference:", Config.MAX_DAYS_DIFF)
print("Tabular features:", TAB_COLS)
print("Applied target transforms:")
for t in Config.TARGET_COLS:
    print(f"  - {t}: {TARGET_TRANSFORMS.get(t, 'none')}")

df.head(3)

Loading valid patches cache from data/river_patches/valid_patches.json
Filtered out 213 patches with completely empty index bands
Applying cloud threshold from metadata field 'patch_cloud_coverage_pct' (threshold=20.0%)...
Filtered out 1554 patches with computed cloud coverage > 20.0%
days_difference range before time filter: min=0.33, max=729.68
Filtered out 4516 patches with days_difference > 120
--------------------------------
Total usable samples: 1154
Unique stations/groups: 134
Targets: ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
Using datetime features: True
Max days difference: 120
Tabular features: ['pet', 'ppt', 'q', 'soil', 'tmax', 'ppt_lag1', 'q_lag1', 'soil_bdod_top30cm_mean', 'soil_cec_top30cm_mean', 'soil_clay_top30cm_mean', 'soil_phh2o_top30cm_mean', 'soil_sand_top30cm_mean', 'soil_soc_top30cm_mean', 'day_sin', 'day_cos']
Applied target transforms:
  - Total Alkalinity: none
  - Electrical Conductance: none
  - Dissolved Reactive Pho

,patch_id,dataset_index,file,lat,lon,target_date,target_date_iso,actual_date,days_difference,cloud_cover,...,tmax,ppt_lag1,q_lag1,soil_bdod_top30cm_mean,soil_cec_top30cm_mean,soil_clay_top30cm_mean,soil_phh2o_top30cm_mean,soil_sand_top30cm_mean,soil_soc_top30cm_mean,group_id
0,203,203,river_patches/patch_203.tif,-27.389722,26.463889,23-02-2011,2011-02-23,2011-02-16,6.667441,2.0,...,29.98,90.9,4.5,124.333333,152.333333,200.666667,66.333333,724.666667,235.000000,-27.3897_26.4639
1,204,204,river_patches/patch_204.tif,-27.398611,26.614722,23-02-2011,2011-02-23,2011-02-16,6.667441,2.0,...,29.77,87.9,4.4,144.666667,140.333333,213.000000,65.333333,695.333333,110.333333,-27.3986_26.6147
2,205,205,river_patches/patch_205.tif,-27.671111,27.236944,23-02-2011,2011-02-23,2011-02-16,6.667441,2.0,...,28.87,77.8,3.9,143.666667,199.000000,263.333333,68.000000,620.333333,120.000000,-27.6711_27.2369


In [5]:
# K-Fold cross-validation setup (location-aware)
LOCATION_ROUND_DECIMALS = 4

df = df.copy()
df["location_id"] = (
    df["lat"].round(LOCATION_ROUND_DECIMALS).astype(str)
    + "_"
    + df["lon"].round(LOCATION_ROUND_DECIMALS).astype(str)
)

gss_outer = GroupShuffleSplit(
    n_splits=1,
    test_size=Config.TEST_SIZE,
    random_state=Config.SEED,
)
train_idx, test_idx = next(gss_outer.split(df, groups=df["location_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# Create inner train/validation splits on training data (by location)
if Config.N_FOLDS == 1:
    gss_inner = GroupShuffleSplit(
        n_splits=1,
        test_size=Config.TEST_SIZE,
        random_state=Config.SEED,
    )
    fold_train_idx, fold_val_idx = next(
        gss_inner.split(train_df, groups=train_df["location_id"])
    )
    fold_splits = [(fold_train_idx, fold_val_idx)]
else:
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    fold_splits = list(gkf.split(train_df, groups=train_df["location_id"]))

# Safety checks: no location leakage across outer split or inner folds
train_locations = set(train_df["location_id"].unique())
test_locations = set(test_df["location_id"].unique())
outer_overlap = train_locations & test_locations
if outer_overlap:
    raise ValueError(
        f"Location leakage in outer split: {len(outer_overlap)} overlapping locations"
    )

for i, (fold_train_idx, fold_val_idx) in enumerate(fold_splits, start=1):
    fold_train_locations = set(train_df.iloc[fold_train_idx]["location_id"].unique())
    fold_val_locations = set(train_df.iloc[fold_val_idx]["location_id"].unique())
    fold_overlap = fold_train_locations & fold_val_locations
    if fold_overlap:
        raise ValueError(
            f"Location leakage in fold {i}: {len(fold_overlap)} overlapping locations"
        )

print(f"Total samples: {len(df)}")
print(f"Train samples (for k-fold): {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Train locations: {train_df['location_id'].nunique()}")
print(f"Test locations:  {test_df['location_id'].nunique()}")
print(f"Number of folds: {Config.N_FOLDS}")
print(f"\nFold sizes:")
for i, (fold_train_idx, fold_val_idx) in enumerate(fold_splits, start=1):
    print(f"  Fold {i}: train={len(fold_train_idx)}, val={len(fold_val_idx)}")

Total samples: 1154
Train samples (for k-fold): 899
Test samples: 255
Train locations: 107
Test locations:  27
Number of folds: 1

Fold sizes:
  Fold 1: train=723, val=176


In [6]:
# K-Fold cross-validation with early stopping
fold_results = []

for fold_idx, (fold_train_idx, fold_val_idx) in enumerate(fold_splits):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{Config.N_FOLDS}")
    print(f"{'='*60}")
    
    # Get fold data
    fold_train_df = train_df.iloc[fold_train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[fold_val_idx].reset_index(drop=True)
    
    print(f"Fold train samples: {len(fold_train_df)}, val samples: {len(fold_val_df)}")
    
    # Fit scalers on THIS fold's training data only
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    
    X_fold_train_tab = x_scaler.fit_transform(fold_train_df[TAB_COLS].values)
    X_fold_val_tab = x_scaler.transform(fold_val_df[TAB_COLS].values)
    X_test_tab = x_scaler.transform(test_df[TAB_COLS].values)
    
    Y_fold_train = y_scaler.fit_transform(fold_train_df[Config.TARGET_COLS].values.astype(np.float32))
    Y_fold_val = y_scaler.transform(fold_val_df[Config.TARGET_COLS].values.astype(np.float32))
    Y_test = y_scaler.transform(test_df[Config.TARGET_COLS].values.astype(np.float32))
    
    # Create datasets and loaders for this fold
    fold_train_ds = WaterQualityDataset(
        fold_train_df, X_fold_train_tab, Y_fold_train, 
        index_bands=Config.INDEX_BANDS,
        img_size=Config.IMG_SIZE, augment=True,
        use_images=Config.USE_IMAGES, use_tabular=Config.USE_TABULAR
    )
    fold_val_ds = WaterQualityDataset(
        fold_val_df, X_fold_val_tab, Y_fold_val,
        index_bands=Config.INDEX_BANDS,
        img_size=Config.IMG_SIZE, augment=False,
        use_images=Config.USE_IMAGES, use_tabular=Config.USE_TABULAR
    )
    test_ds = WaterQualityDataset(
        test_df, X_test_tab, Y_test,
        index_bands=Config.INDEX_BANDS,
        img_size=Config.IMG_SIZE, augment=False,
        use_images=Config.USE_IMAGES, use_tabular=Config.USE_TABULAR
    )
    
    fold_train_loader = DataLoader(
        fold_train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, 
        num_workers=Config.NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=True
    )
    fold_val_loader = DataLoader(
        fold_val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, 
        num_workers=Config.NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    test_loader = DataLoader(
        test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, 
        num_workers=Config.NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    
    print(f"Created datasets and loaders for fold {fold_idx + 1}")
    
    # Create model for this fold
    model = MultiModalResNetRegressor(
        tab_dim=len(TAB_COLS),
        out_dim=len(Config.TARGET_COLS),
        index_bands_count=len(Config.INDEX_BANDS),
        use_images=Config.USE_IMAGES,
        use_tabular=Config.USE_TABULAR,
        tab_dropout=Config.TAB_DROPOUT,
        head_dropout=Config.HEAD_DROPOUT
    ).to(device)
    
    print(f"Model: {model.__class__.__name__}")
    
    # Train with early stopping
    train_result = train_with_early_stopping(
        model=model,
        train_loader=fold_train_loader,
        val_loader=fold_val_loader,
        test_loader=test_loader,
        device=device,
        max_epochs=Config.MAX_EPOCHS,
        patience=Config.PATIENCE,
        lr=Config.LR,
        weight_decay=Config.WEIGHT_DECAY
    )
    
    print(f"Fold {fold_idx + 1} best test loss: {train_result['best_test_loss']:.5f}")
    
    # Store results
    fold_results.append({
        "fold": fold_idx + 1,
        "model_state": train_result["best_state"],
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "history": train_result["history"],
        "best_test_loss": train_result["best_test_loss"]
    })

print(f"\n{'='*60}")
print("K-FOLD CROSS-VALIDATION COMPLETE")
print(f"{'='*60}")


FOLD 1/1
Fold train samples: 723, val samples: 176
Created datasets and loaders for fold 1
Model: MultiModalResNetRegressor


Epoch 01/40 | train=0.3970 | val=0.3632


Epoch 02/40 | train=0.3349 | val=0.3495


Epoch 03/40 | train=0.2850 | val=0.3520


Epoch 04/40 | train=0.2505 | val=0.3918


Epoch 05/40 | train=0.2458 | val=0.3710


... (12 outputs left out)

Epoch 18/40 | train=0.1603 | val=0.2607


Epoch 19/40 | train=0.1573 | val=0.2616


Epoch 20/40 | train=0.1700 | val=0.2548
Early stopping after 20 epochs (val loss not improving)


Fold 1 best test loss: 0.32295

K-FOLD CROSS-VALIDATION COMPLETE


In [7]:
# Evaluate all folds on test set
all_metrics = []

for fold_result in fold_results:
    print(f"Evaluating Fold {fold_result['fold']}...", end=' ')
    
    # Recreate model and load weights
    model = MultiModalResNetRegressor(
        tab_dim=len(TAB_COLS),
        out_dim=len(Config.TARGET_COLS),
        index_bands_count=len(Config.INDEX_BANDS),
        use_images=Config.USE_IMAGES,
        use_tabular=Config.USE_TABULAR,
        tab_dropout=Config.TAB_DROPOUT,
        head_dropout=Config.HEAD_DROPOUT
    ).to(device)
    model.load_state_dict(fold_result["model_state"])
    
    # Scale test data using this fold's scalers
    X_test_tab = fold_result["x_scaler"].transform(test_df[TAB_COLS].values)
    Y_test = fold_result["y_scaler"].transform(test_df[Config.TARGET_COLS].values.astype(np.float32))
    
    # Create test dataset and loader
    test_ds = WaterQualityDataset(
        test_df, X_test_tab, Y_test,
        index_bands=Config.INDEX_BANDS,
        img_size=Config.IMG_SIZE, augment=False,
        use_images=Config.USE_IMAGES, use_tabular=Config.USE_TABULAR
    )
    test_loader = DataLoader(
        test_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    
    # Get predictions
    pred_scaled, true_scaled = predict(model, test_loader, device)
    
    # Inverse standardization to transformed target space
    pred = fold_result["y_scaler"].inverse_transform(pred_scaled)
    true = fold_result["y_scaler"].inverse_transform(true_scaled)
    
    # Inverse per-target transforms back to original target scale
    for i, t in enumerate(Config.TARGET_COLS):
        transform_name = TARGET_TRANSFORMS.get(t, "none")
        if transform_name == "log1p":
            pred[:, i] = np.expm1(pred[:, i])
            true[:, i] = np.expm1(true[:, i])
        elif transform_name != "none":
            raise ValueError(f"Unsupported transform '{transform_name}' for target '{t}'")
    
    # Calculate metrics for each target
    for i, t in enumerate(Config.TARGET_COLS):
        mae = mean_absolute_error(true[:, i], pred[:, i])
        rmse = math.sqrt(mean_squared_error(true[:, i], pred[:, i]))
        r2 = r2_score(true[:, i], pred[:, i])
        all_metrics.append({
            "fold": fold_result['fold'],
            "target": t,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })
    print("✓")

# Create DataFrame with all metrics
metrics_df = pd.DataFrame(all_metrics)

# Display individual fold results
print("\n" + "="*70)
print("INDIVIDUAL FOLD METRICS ON TEST SET")
print("="*70)
display(metrics_df)

# Calculate and display mean and std across folds for each target
print("\n" + "="*70)
print("AVERAGE METRICS ACROSS FOLDS")
print("="*70)
summary_metrics = metrics_df.groupby('target')[['MAE', 'RMSE', 'R2']].agg(['mean', 'std'])
display(summary_metrics)

metrics_df

Evaluating Fold 1... 

✓

INDIVIDUAL FOLD METRICS ON TEST SET


,fold,target,MAE,RMSE,R2
0,1,Total Alkalinity,40.404907,54.459289,0.298613
1,1,Electrical Conductance,186.261826,292.013765,0.386118
2,1,Dissolved Reactive Phosphorus,42.898048,68.064286,-0.226375



AVERAGE METRICS ACROSS FOLDS


MAE            RMSE            R2    
                                     mean std        mean std      mean std
target                                                                     
Dissolved Reactive Phosphorus   42.898048 NaN   68.064286 NaN -0.226375 NaN
Electrical Conductance         186.261826 NaN  292.013765 NaN  0.386118 NaN
Total Alkalinity                40.404907 NaN   54.459289 NaN  0.298613 NaN

,fold,target,MAE,RMSE,R2
0,1,Total Alkalinity,40.404907,54.459289,0.298613
1,1,Electrical Conductance,186.261826,292.013765,0.386118
2,1,Dissolved Reactive Phosphorus,42.898048,68.064286,-0.226375


In [8]:
# Optional: Save one of the models for later use
# Uncomment to save

# import pickle
# from pathlib import Path
# 
# artifacts_dir = Path("model_artifacts")
# artifacts_dir.mkdir(exist_ok=True)
# 
# # Save first fold's model as an example
# torch.save(fold_results[0]["model_state"], artifacts_dir / "model_fold1.pt")
# 
# with open(artifacts_dir / "x_scaler_fold1.pkl", "wb") as f:
#     pickle.dump(fold_results[0]["x_scaler"], f)
# with open(artifacts_dir / "y_scaler_fold1.pkl", "wb") as f:
#     pickle.dump(fold_results[0]["y_scaler"], f)
# 
# print("Saved model + scalers to", artifacts_dir)

In [9]:
# Inference on submission template (no cloud/empty-patch filtering)
from pathlib import Path

FOLD_FOR_INFERENCE = 1  # Select fold: 1 to Config.N_FOLDS

sub_df, sub_tab_cols = load_submission_for_inference(
    submission_csv_path=Path("data/submission_template.csv"),
    meta_path=Config.META_PATH,
    idx_dir=Config.IDX_DIR,
    index_bands=Config.INDEX_BANDS,
    soilgrids_csv_path=Config.SOILGRIDS_CSV_PATH,
    tc_features_csv_path=Config.TC_FEATURES_ALL_CSV_PATH,
    use_datetime_features=Config.USE_DATETIME_FEATURES,
)

# Ensure tab cols match training (same order as TAB_COLS)
sub_tab_cols = [c for c in TAB_COLS if c in sub_tab_cols]
for c in TAB_COLS:
    if c not in sub_df.columns:
        sub_df[c] = 0.0
sub_tab_cols = TAB_COLS

fold_result = fold_results[FOLD_FOR_INFERENCE - 1]
X_sub_tab = fold_result["x_scaler"].transform(sub_df[sub_tab_cols].values)
Y_dummy = np.zeros((len(sub_df), len(Config.TARGET_COLS)), dtype=np.float32)

sub_ds = WaterQualityDataset(
    sub_df, X_sub_tab, Y_dummy,
    index_bands=Config.INDEX_BANDS,
    img_size=Config.IMG_SIZE, augment=False,
    use_images=Config.USE_IMAGES, use_tabular=Config.USE_TABULAR
)
sub_loader = DataLoader(
    sub_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
    num_workers=Config.NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

model = MultiModalResNetRegressor(
    tab_dim=len(TAB_COLS),
    out_dim=len(Config.TARGET_COLS),
    index_bands_count=len(Config.INDEX_BANDS),
    use_images=Config.USE_IMAGES, use_tabular=Config.USE_TABULAR,
    tab_dropout=Config.TAB_DROPOUT, head_dropout=Config.HEAD_DROPOUT
).to(device)
model.load_state_dict(fold_result["model_state"])

pred_scaled, _ = predict(model, sub_loader, device)
pred = fold_result["y_scaler"].inverse_transform(pred_scaled)

for i, t in enumerate(Config.TARGET_COLS):
    if TARGET_TRANSFORMS.get(t, "none") == "log1p":
        pred[:, i] = np.expm1(pred[:, i])

# Add predictions to matched rows
for i, t in enumerate(Config.TARGET_COLS):
    sub_df[t] = pred[:, i]

# Load original submission template and merge predictions
submission_template = pd.read_csv("data/submission_template.csv")
submission_with_preds = submission_template.copy()

# Create merge keys
sub_df["merge_key"] = (
    sub_df["Latitude"].round(5).astype(str) + "_" +
    sub_df["Longitude"].round(5).astype(str) + "_" +
    pd.to_datetime(sub_df["Sample Date"], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
)
submission_with_preds["merge_key"] = (
    submission_with_preds["Latitude"].round(5).astype(str) + "_" +
    submission_with_preds["Longitude"].round(5).astype(str) + "_" +
    pd.to_datetime(submission_with_preds["Sample Date"], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
)

# Merge predictions (left join to keep all submission rows)
pred_df = sub_df[["merge_key"] + Config.TARGET_COLS].copy()
submission_with_preds = submission_with_preds.drop(columns=Config.TARGET_COLS).merge(
    pred_df, on="merge_key", how="left"
)
submission_with_preds = submission_with_preds.drop(columns=["merge_key"])

# Save to root folder
output_path = f"submission_fold{FOLD_FOR_INFERENCE}.csv"
submission_with_preds.to_csv(output_path, index=False)

print(f"Inference complete: {len(sub_df)} matched rows out of {len(submission_template)} total (fold {FOLD_FOR_INFERENCE})")
print(f"Saved to: {output_path}")
print(f"Rows with predictions: {submission_with_preds[Config.TARGET_COLS[0]].notna().sum()}")
print(f"Rows without predictions: {submission_with_preds[Config.TARGET_COLS[0]].isna().sum()}")
display(submission_with_preds.head(10))

Inference complete: 200 matched rows out of 200 total (fold 1)
Saved to: submission_fold1.csv
Rows with predictions: 200
Rows without predictions: 0


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,136.350342,439.204529,21.113558
1,-33.329167,26.077500,16-09-2015,90.573685,426.822510,17.634892
2,-32.991639,27.640028,07-05-2015,107.966354,422.408081,23.923027
3,-34.096389,24.439167,07-02-2012,74.481285,315.510803,20.815083
4,-32.000556,28.581667,01-10-2014,141.719345,587.190796,27.307878
5,-32.086390,25.575560,19-07-2013,175.605270,713.696655,81.040710
6,-32.000556,28.581667,03-09-2014,134.546097,547.423279,16.758198
7,-32.991639,27.640028,02-10-2014,130.173203,498.719788,16.867449
8,-32.000556,28.581667,06-08-2014,76.519150,342.416809,15.435361
9,-33.185361,27.390750,22-09-2011,178.115356,707.344971,148.025299
